<a href="https://colab.research.google.com/github/thamiherzing/BillRetrieval/blob/main/Bill_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

excel_files = list(Path("/content").glob("*.xlsx"))

print("Excel files uploaded:")
for file in excel_files:
    print(file.name)


Excel files uploaded:


In [ ]:
import pandas as pd

bills = pd.read_excel("/content/Bills 1.xlsx")

display(bills.head())
print("Column names:", bills.columns.tolist())

,Bill_ID,Unnamed: 1,Introduced,Name,Sponsor,Sponsor Party,Committees
0,USA_0001,H.R.10167,2026-08-27,Common Cents Act,"McClain, Lisa C. [Rep.-R-MI-9]",Republican,"House - Financial Services | Senate - Banking,..."
1,USA_0002,H.R.9770,2026-07-18,"Continuing Appropriations Act, 2027","Cole, Tom [Rep.-R-OK-4]",Republican,House - Appropriations; Budget
2,USA_0003,H.R.9723,2026-07-16,Fit Future Act,"Mackenzie, Ryan [Rep.-R-PA-7]",Republican,House - Education and Workforce | Senate - Hea...
3,USA_0004,H.R.9617,2026-07-09,CHARM Act,"Palmer, Gary J. [Rep.-R-AL-6]",Republican,House - Energy and Commerce | Senate - Environ...
4,USA_0005,H.R.9616,2026-07-09,EMRTAI Authorization Act of 2026,"Pfluger, August [Rep.-R-TX-11]",Republican,House - Energy and Commerce | Senate - Environ...


Column names: ['Bill_ID', 'Unnamed: 1', 'Introduced', 'Name', 'Sponsor', 'Sponsor Party', 'Committees']


In [2]:
# Rename column B, which had no heading in Excel
bills = bills.rename(columns={"Unnamed: 1": "Bill_Number"})

# Keep only rows containing a bill number
bills = bills[
    bills["Bill_Number"].notna()
    & bills["Bill_Number"].astype(str).str.strip().ne("")
].copy()

# Extract the number from entries such as H.R.10167
bill_numbers = bills["Bill_Number"].astype(str).str.extract(
    r"^\s*H\s*\.\s*R\s*\.\s*(\d+)\s*$",
    expand=False
)

# Stop if any entry does not match the expected format
if bill_numbers.isna().any():
    display(bills.loc[bill_numbers.isna(), ["Bill_ID", "Bill_Number"]])
    raise ValueError("Check the bill numbers displayed above.")

# Create the identifier: Congress 119 + House bill + number
bills["Congress_Bill_ID"] = "119-hr-" + bill_numbers

display(bills[["Bill_ID", "Bill_Number", "Congress_Bill_ID"]])
print("Total bills:", len(bills))
print("Duplicate congressional IDs:", bills["Congress_Bill_ID"].duplicated().sum())


NameError: name 'bills' is not defined

In [ ]:
from getpass import getpass

congress_api_key = getpass("RfNGmBMCjTwfDjFptcjhgCqhcPq8z6tRBnPRTHzQ").strip()

if not congress_api_key:
    raise ValueError("No key entered. Run this cell again.")

print("Key stored for this Colab session. Ready to test the connection.")

RfNGmBMCjTwfDjFptcjhgCqhcPq8z6tRBnPRTHzQ··········
Key stored for this Colab session. Ready to test the connection.


In [ ]:
import requests

response = requests.get(
    "https://api.congress.gov/v3/bill/119/hr/10167",
    params={"api_key": congress_api_key, "format": "json"},
    timeout=30
)

print("Connection status:", response.status_code)

if response.status_code == 200:
    bill_info = response.json()["bill"]
    print("Bill:", bill_info["type"], bill_info["number"])
    print("Title:", bill_info["title"])
else:
    print("The request did not succeed. Send me the status number.")

Connection status: 200
Bill: HR 10167
Title: Common Cents Act


In [ ]:
actions = []
offset = 0

while True:
    response = requests.get(
        "https://api.congress.gov/v3/bill/119/hr/10167/actions",
        params={
            "api_key": congress_api_key,
            "format": "json",
            "limit": 250,
            "offset": offset
        },
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Request failed with status {response.status_code}"
        )

    data = response.json()
    batch = data.get("actions", [])
    actions.extend(batch)

    if not data.get("pagination", {}).get("next"):
        break

    if not batch:
        raise RuntimeError("An empty page was returned unexpectedly.")

    offset += len(batch)

actions_table = pd.DataFrame([
    {
        "Date": action.get("actionDate"),
        "Action": action.get("text"),
        "Record references": action.get("links", [])
    }
    for action in actions
])

display(actions_table.sort_values("Date"))
print("Total actions retrieved:", len(actions))

,Date,Action,Record references
16,2026-08-27,Introduced in House,[]
14,2026-08-27,Referred to the House Committee on Financial S...,[]
15,2026-08-27,Introduced in House,[]
13,2026-09-14,Ms. De La Cruz moved to suspend the rules and ...,[]
12,2026-09-14,Considered under suspension of the rules. (con...,[]
11,2026-09-14,DEBATE - The House proceeded with forty minute...,[]
10,2026-09-14,Passed/agreed to in House: On motion to suspen...,[]
9,2026-09-14,On motion to suspend the rules and pass the bi...,[]
8,2026-09-14,Motion to reconsider laid on the table Agreed ...,[]
7,2026-09-15,Received in the Senate and Read twice and refe...,[]


Total actions retrieved: 17


In [ ]:
for action in actions:
    if action.get("actionDate") == "2026-09-14":
        print(action.get("text", ""))
        print()

Motion to reconsider laid on the table Agreed to without objection.

On motion to suspend the rules and pass the bill Agreed to by voice vote. (text: CR H5599-5601)

Passed/agreed to in House: On motion to suspend the rules and pass the bill Agreed to by voice vote. (text: CR H5599-5601)

DEBATE - The House proceeded with forty minutes of debate on H.R. 10167.

Considered under suspension of the rules. (consideration: CR H5599-5602)

Ms. De La Cruz moved to suspend the rules and pass the bill.



In [ ]:
record_sections = []
offset = 0

while True:
    response = requests.get(
        "https://api.govinfo.gov/packages/CREC-2026-09-14/granules",
        params={
            "api_key": congress_api_key,
            "pageSize": 100,
            "offset": offset
        },
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"GovInfo request failed: {response.status_code}"
        )

    data = response.json()
    batch = data.get("granules", [])
    record_sections.extend(batch)

    if not data.get("nextPage"):
        break

    if not batch:
        raise RuntimeError("An empty page was returned unexpectedly.")

    offset += len(batch)

matches = [
    section for section in record_sections
    if "common cents" in section.get("title", "").lower()
]

print("Matching sections:", len(matches))

for section in matches:
    print("Title:", section.get("title"))
    print("Section ID:", section.get("granuleId"))
    print()


Matching sections: 1
Title: COMMON CENTS ACT
Section ID: CREC-2026-09-14-pt1-PgH5599-2



In [ ]:
from bs4 import BeautifulSoup
from pathlib import Path

section_id = matches[0]["granuleId"]

source_url = (
    "https://www.govinfo.gov/content/pkg/CREC-2026-09-14/html/"
    f"{section_id}.htm"
)

response = requests.get(source_url, timeout=30)
response.raise_for_status()

# Extract the Record text from the webpage
soup = BeautifulSoup(response.content, "html.parser")
record_body = soup.find("pre")

if record_body is None:
    raise ValueError("Expected Record text was not found. Stop and check the page.")

record_text = record_body.get_text()

if "COMMON CENTS ACT" not in record_text.upper():
    raise ValueError("The downloaded section does not match the expected title.")

# Save the original webpage and extracted text in Colab
output_folder = Path("/content/congressional_record")
output_folder.mkdir(exist_ok=True)

(output_folder / f"{section_id}.html").write_bytes(response.content)
(output_folder / f"{section_id}.txt").write_text(
    record_text, encoding="utf-8"
)

print("Source:", source_url)
print("Characters downloaded:", len(record_text))
print("\nPreview:\n")
print(record_text[:3500])

Source: https://www.govinfo.gov/content/pkg/CREC-2026-09-14/html/CREC-2026-09-14-pt1-PgH5599-2.htm
Characters downloaded: 25801

Preview:


[Congressional Record Volume 172, Number 144 (Monday, September 14, 2026)]
[House]
[Pages H5599-H5602]
From the Congressional Record Online through the Government Publishing Office [www.gpo.gov]




                              {time}  1910
                            COMMON CENTS ACT

  Ms. DE LA CRUZ. Mr. Speaker, I move to suspend the rules and pass the 
bill (H.R. 10167) to direct the Secretary of the Treasury to stop 
minting the penny, to require cash transactions to be rounded up or 
down to the nearest 5 cents, and for other purposes.
  The Clerk read the title of the bill.
  The text of the bill is as follows:

                               H.R. 10167

       Be it enacted by the Senate and House of Representatives of 
     the United States of America in Congress assembled,

     SECTION 1. SHORT TITLE.

       This Act may be cited as 

In [ ]:
import re

# Find likely speaker labels at the beginning of a line
speaker_pattern = re.compile(
    r"^[ \t]*(?P<speaker>"
    r"(?:Mr\.|Ms\.|Mrs\.)[ \t]+[A-Z][A-Z '\u2019.—-]*"
    r"(?:[ \t]+of[ \t]+[A-Za-z]+(?:[ \t]+[A-Za-z]+)*)?"
    r"|The SPEAKER(?: pro tempore)?(?:[ \t]+\([^\n)]*\))?"
    r"|The CHAIR(?: pro tempore)?"
    r")\.[ \t]+",
    re.MULTILINE
)

speaker_matches = list(speaker_pattern.finditer(record_text))

for number, match in enumerate(speaker_matches, start=1):
    preview = record_text[match.end():match.end() + 180]
    preview = " ".join(preview.split())

    print(f"{number}. {match.group('speaker')}")
    print(f"   {preview}\n")

print("Potential speaker turns:", len(speaker_matches))

1. Ms. DE LA CRUZ
   Mr. Speaker, I move to suspend the rules and pass the bill (H.R. 10167) to direct the Secretary of the Treasury to stop minting the penny, to require cash transactions to be roun

2. The SPEAKER pro tempore (Mr. Hurd of Colorado)
   Pursuant to the rule, the gentlewoman from Texas (Ms. De La Cruz) and the gentleman from California (Mr. Sherman) each will control 20 minutes. The Chair recognizes the gentlew

3. Ms. DE LA CRUZ
   Mr. Speaker, I ask unanimous consent that all Members may have 5 legislative days to revise and extend their remarks and to include extraneous material on the bill. The SPEAKER

4. The SPEAKER pro tempore
   Is there objection to the request of the gentlewoman from Texas? There was no objection. Ms. DE LA CRUZ. Mr. Speaker, I yield myself such time as I may consume. Mr. Speaker

5. Ms. DE LA CRUZ
   Mr. Speaker, I yield myself such time as I may consume. Mr. Speaker, I include in the Record the CBO estimate for this bill. EFFECTS ON DIRECT S

In [ ]:
# Allow mixed-case surnames such as McCLAIN
speaker_pattern = re.compile(
    r"^[ \t]*(?P<speaker>"
    r"(?:Mr\.|Ms\.|Mrs\.)[ \t]+[A-Z][A-Za-z '\u2019’—-]*"
    r"|The SPEAKER(?: pro tempore)?(?:[ \t]+\([^\n)]*\))?"
    r"|The CHAIR(?: pro tempore)?"
    r")\.[ \t]+",
    re.MULTILINE
)

speaker_matches = list(speaker_pattern.finditer(record_text))

# Each candidate turn ends where the next speaker label begins
candidate_turns = []

for index, match in enumerate(speaker_matches):
    end = (
        speaker_matches[index + 1].start()
        if index + 1 < len(speaker_matches)
        else len(record_text)
    )

    candidate_turns.append({
        "turn_number": index + 1,
        "speaker_raw": match.group("speaker"),
        "text": record_text[match.end():end].strip()
    })

for turn in candidate_turns:
    preview = " ".join(turn["text"].split())[:220]
    print(f"{turn['turn_number']}. {turn['speaker_raw']}")
    print(f"   {preview}\n")

print("Potential speaker turns:", len(candidate_turns))


1. Ms. DE LA CRUZ
   Mr. Speaker, I move to suspend the rules and pass the bill (H.R. 10167) to direct the Secretary of the Treasury to stop minting the penny, to require cash transactions to be rounded up or down to the nearest 5 cents, and

2. The SPEAKER pro tempore (Mr. Hurd of Colorado)
   Pursuant to the rule, the gentlewoman from Texas (Ms. De La Cruz) and the gentleman from California (Mr. Sherman) each will control 20 minutes. The Chair recognizes the gentlewoman from Texas. General Leave

3. Ms. DE LA CRUZ
   Mr. Speaker, I ask unanimous consent that all Members may have 5 legislative days to revise and extend their remarks and to include extraneous material on the bill.

4. The SPEAKER pro tempore
   Is there objection to the request of the gentlewoman from Texas? There was no objection.

5. Ms. DE LA CRUZ
   Mr. Speaker, I yield myself such time as I may consume. Mr. Speaker, I include in the Record the CBO estimate for this bill. EFFECTS ON DIRECT SPENDING AND REVENUES OF 

In [ ]:
review_turn_numbers = [6, 7, 9, 11, 12]

for turn in candidate_turns:
    if turn["turn_number"] in review_turn_numbers:
        print("=" * 80)
        print(f"TURN {turn['turn_number']} — {turn['speaker_raw']}")
        print("=" * 80)
        print(turn["text"])
        print()


TURN 6 — Ms. DE LA CRUZ
Mr. Speaker, I rise today in support of Republican 
Conference Chairwoman McClain's Common Cents Act. Rather than spending 
more money producing a coin than the coin itself is worth, this 
straightforward, commonsense bill ensures that Congress is a good 
steward of taxpayers' dollars by providing long-term fiscally 
responsible solutions.
  Mr. Speaker, let's look at the numbers. According to an audit of the 
United States Mint, it costs $3.69 to make a 1-cent penny. In fact, for 
19 consecutive years, the U.S. Government has spent more than one penny 
to make one penny. In 2024 alone, taxpayers lost more than $85 million 
producing pennies.
  Chairwoman McClain's bill eliminates that wasteful spending by 
codifying President Trump's directive to the Mint to stop producing new 
pennies for circulation. However, we also need to make sure we are not 
solving one problem by creating another.
  If we stop making pennies, then Americans are likely to use more 
nicke

In [ ]:
def clean_record_text(text):
    # Remove editorial page and time markers
    text = re.sub(r"\[\[Page [^\]]+\]\]", "", text)
    text = re.sub(r"\{time\}\s*\d{3,4}", "", text)

    # Join line breaks and repeated spaces
    return re.sub(r"\s+", " ", text).strip()


pilot_speeches = pd.DataFrame([
    {
        "speech_id": f"119-hr-10167_2026-09-14_t{turn['turn_number']:02d}",
        "Bill_ID": "USA_0001",
        "Congress_Bill_ID": "119-hr-10167",
        "date": "2026-09-14",
        "chamber": "House",
        "turn_number": turn["turn_number"],
        "speaker_raw": turn["speaker_raw"],
        "speech_text_raw": turn["text"],
        "speech_text_clean": clean_record_text(turn["text"]),
        "source_url": source_url,
        "document_id": section_id
    }
    for turn in candidate_turns
    if turn["turn_number"] in [6, 7, 9, 11, 12]
])

display(
    pilot_speeches[
        ["speech_id", "speaker_raw", "turn_number"]
    ]
)

print("Interventions retained:", len(pilot_speeches))

,speech_id,speaker_raw,turn_number
0,119-hr-10167_2026-09-14_t06,Ms. DE LA CRUZ,6
1,119-hr-10167_2026-09-14_t07,Mr. SHERMAN of California,7
2,119-hr-10167_2026-09-14_t09,Mrs. McCLAIN,9
3,119-hr-10167_2026-09-14_t11,Mr. FOSTER,11
4,119-hr-10167_2026-09-14_t12,Mrs. McCLAIN,12


Interventions retained: 5


In [ ]:
# Preserve every candidate turn and record the pilot selection
pilot_audit = pd.DataFrame(candidate_turns)

pilot_audit["retained_in_pilot"] = pilot_audit["turn_number"].isin(
    pilot_speeches["turn_number"]
)

pilot_audit["document_id"] = section_id
pilot_audit["source_url"] = source_url

# Save both tables in one workbook
output_path = "/content/Common_Cents_pilot.xlsx"

with pd.ExcelWriter(output_path) as writer:
    pilot_speeches.to_excel(
        writer, sheet_name="Speeches", index=False
    )
    pilot_audit.to_excel(
        writer, sheet_name="All_candidate_turns", index=False
    )

# Download the workbook to your computer
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import time
from pathlib import Path

history_folder = Path("/content/bill_histories")
history_folder.mkdir(exist_ok=True)

collection_log = []

for bill_id in bills["Congress_Bill_ID"]:
    output_file = history_folder / f"{bill_id}.json"

    # Reuse a completed download if this cell is run again
    if output_file.exists():
        print(f"Already saved: {bill_id}")
        collection_log.append({
            "Congress_Bill_ID": bill_id,
            "status": "already_saved"
        })
        continue

    congress, bill_type, bill_number = bill_id.split("-")
    bill_actions = []
    offset = 0

    try:
        while True:
            response = requests.get(
                f"https://api.congress.gov/v3/bill/"
                f"{congress}/{bill_type}/{bill_number}/actions",
                params={
                    "api_key": congress_api_key,
                    "format": "json",
                    "limit": 250,
                    "offset": offset
                },
                timeout=30
            )

            if response.status_code != 200:
                raise RuntimeError(
                    f"HTTP status {response.status_code}"
                )

            data = response.json()

            if "actions" not in data:
                raise RuntimeError("Response has no actions field")

            batch = data["actions"]
            bill_actions.extend(batch)

            if not data.get("pagination", {}).get("next"):
                break

            if not batch:
                raise RuntimeError("Unexpected empty page")

            offset += len(batch)
            time.sleep(0.5)

        # Save only after every page has been retrieved
        output_file.write_text(
            json.dumps(
                {
                    "Congress_Bill_ID": bill_id,
                    "actions": bill_actions
                },
                ensure_ascii=False,
                indent=2
            ),
            encoding="utf-8"
        )

        collection_log.append({
            "Congress_Bill_ID": bill_id,
            "status": "saved",
            "action_count": len(bill_actions)
        })

        print(f"Saved: {bill_id} — {len(bill_actions)} actions")

    except Exception as error:
        # Avoid displaying request URLs that could contain the API key
        reason = (
            str(error)
            if isinstance(error, RuntimeError)
            else type(error).__name__
        )

        collection_log.append({
            "Congress_Bill_ID": bill_id,
            "status": "failed",
            "reason": reason
        })

        print(f"Failed: {bill_id} — {reason}")

    time.sleep(0.5)

collection_log = pd.DataFrame(collection_log)
collection_log.to_csv(
    "/content/bill_history_collection_log.csv",
    index=False
)

print("\nCollection summary:")
print(collection_log["status"].value_counts())

display(collection_log[collection_log["status"] == "failed"])


Saved: 119-hr-10167 — 17 actions
Saved: 119-hr-9770 — 16 actions
Saved: 119-hr-9723 — 12 actions
Saved: 119-hr-9617 — 19 actions
Saved: 119-hr-9616 — 19 actions
Saved: 119-hr-9500 — 14 actions
Saved: 119-hr-9497 — 19 actions
Saved: 119-hr-8897 — 19 actions
Saved: 119-hr-8884 — 20 actions
Saved: 119-hr-8882 — 15 actions
Saved: 119-hr-8880 — 15 actions
Saved: 119-hr-8872 — 15 actions
Saved: 119-hr-8790 — 18 actions
Saved: 119-hr-8770 — 17 actions
Saved: 119-hr-8658 — 21 actions
Saved: 119-hr-8646 — 23 actions
Saved: 119-hr-8552 — 17 actions
Saved: 119-hr-8029 — 18 actions
Saved: 119-hr-7726 — 22 actions
Saved: 119-hr-7567 — 70 actions
Saved: 119-hr-7436 — 18 actions
Saved: 119-hr-7396 — 15 actions
Saved: 119-hr-7305 — 16 actions
Saved: 119-hr-6956 — 15 actions
Saved: 119-hr-6893 — 18 actions
Saved: 119-hr-6618 — 16 actions
Saved: 119-hr-6556 — 16 actions
Saved: 119-hr-6481 — 17 actions
Saved: 119-hr-6480 — 17 actions
Saved: 119-hr-6338 — 13 actions
Saved: 119-hr-6332 — 16 actions
Saved: 

,Congress_Bill_ID,status,action_count


In [ ]:
import re

candidate_rows = []

for bill_id in bills["Congress_Bill_ID"]:
    saved_history = json.loads(
        (history_folder / f"{bill_id}.json").read_text(
            encoding="utf-8"
        )
    )

    for action in saved_history["actions"]:
        text = action.get("text", "")
        chamber = action.get("actionBy", "")

        # House Congressional Record references, such as H5599-5602
        house_pages = re.findall(
            r"\bH\d+(?:\s*[-–]\s*H?\d+)?\b",
            text
        )

        # Explicit descriptions of House debate
        explicit_house_debate = bool(
            re.search(
                r"\bHouse\b.*\b(?:debate|consideration)\b",
                text,
                flags=re.IGNORECASE
            )
        )

        # Other debate/consideration actions explicitly attributed to House
        house_action = (
            chamber.lower() == "house"
            and bool(re.search(
                r"\b(?:debate|considered|consideration)\b",
                text,
                flags=re.IGNORECASE
            ))
        )

        if house_pages or explicit_house_debate or house_action:
            candidate_rows.append({
                "Congress_Bill_ID": bill_id,
                "date": action.get("actionDate"),
                "action_text": text,
                "House_pages": "; ".join(house_pages)
            })

candidate_actions = pd.DataFrame(
    candidate_rows,
    columns=[
        "Congress_Bill_ID", "date",
        "action_text", "House_pages"
    ]
).drop_duplicates()

# One row per bill and candidate date
candidate_dates = (
    candidate_actions[["Congress_Bill_ID", "date"]]
    .drop_duplicates()
    .sort_values(["Congress_Bill_ID", "date"])
    .reset_index(drop=True)
)

# Identify bills needing a separate search
bills_without_dates = bills.loc[
    ~bills["Congress_Bill_ID"].isin(
        candidate_dates["Congress_Bill_ID"]
    ),
    ["Bill_ID", "Congress_Bill_ID", "Name"]
]

candidate_actions.to_csv(
    "/content/House_candidate_actions.csv", index=False
)
candidate_dates.to_csv(
    "/content/House_candidate_dates.csv", index=False
)

display(candidate_dates)

print("Bills with candidate dates:",
      candidate_dates["Congress_Bill_ID"].nunique())
print("Bill-date combinations:", len(candidate_dates))
print("Bills needing a separate search:", len(bills_without_dates))

display(bills_without_dates)

,Congress_Bill_ID,date
0,119-hr-10167,2026-09-14
1,119-hr-6007,2026-07-15
2,119-hr-6020,2026-04-14
3,119-hr-6162,2026-06-02
4,119-hr-6238,2026-07-20
5,119-hr-6310,2026-04-14
6,119-hr-6332,2026-07-13
7,119-hr-6338,2026-06-08
8,119-hr-6480,2026-03-24
9,119-hr-6481,2026-03-24


Bills with candidate dates: 36
Bill-date combinations: 42
Bills needing a separate search: 0


,Bill_ID,Congress_Bill_ID,Name


In [ ]:
section_folder = Path("/content/record_section_lists")
section_folder.mkdir(exist_ok=True)

section_rows = []
section_errors = []

dates = sorted(candidate_dates["date"].dropna().unique())

for date in dates:
    saved_file = section_folder / f"{date}.json"

    try:
        if saved_file.exists():
            sections = json.loads(
                saved_file.read_text(encoding="utf-8")
            )
        else:
            sections = []
            offset = 0

            while True:
                response = requests.get(
                    f"https://api.govinfo.gov/packages/"
                    f"CREC-{date}/granules",
                    params={
                        "api_key": congress_api_key,
                        "pageSize": 100,
                        "offset": offset
                    },
                    timeout=30
                )

                if response.status_code != 200:
                    raise RuntimeError(
                        f"HTTP status {response.status_code}"
                    )

                data = response.json()

                if "granules" not in data:
                    raise RuntimeError("No section list in response")

                batch = data["granules"]
                sections.extend(batch)

                if not data.get("nextPage"):
                    break

                if not batch:
                    raise RuntimeError("Unexpected empty page")

                offset += len(batch)
                time.sleep(0.5)

            saved_file.write_text(
                json.dumps(sections, ensure_ascii=False, indent=2),
                encoding="utf-8"
            )

        for section in sections:
            section_rows.append({
                "date": date,
                "document_id": section.get("granuleId"),
                "title": section.get("title")
            })

        print(f"{date}: {len(sections)} sections")

    except Exception as error:
        reason = (
            str(error)
            if isinstance(error, RuntimeError)
            else type(error).__name__
        )
        section_errors.append({"date": date, "reason": reason})
        print(f"{date}: FAILED — {reason}")

    time.sleep(0.5)

record_index = pd.DataFrame(
    section_rows,
    columns=["date", "document_id", "title"]
).drop_duplicates()

record_index.to_csv("/content/record_index.csv", index=False)

print("\nDates requested:", len(dates))
print("Dates retrieved:", record_index["date"].nunique())
print("Sections indexed:", len(record_index))
print("Failed dates:", len(section_errors))

if section_errors:
    display(pd.DataFrame(section_errors))


2026-03-24: 189 sections
2026-03-26: 199 sections
2026-04-14: 204 sections
2026-04-27: 211 sections
2026-04-29: 180 sections
2026-04-30: 248 sections
2026-06-02: 143 sections
2026-06-03: 243 sections
2026-06-04: 453 sections
2026-06-08: 210 sections
2026-06-23: 294 sections
2026-06-29: 141 sections
2026-06-30: 166 sections
2026-07-13: 294 sections
2026-07-14: 269 sections
2026-07-15: 175 sections
2026-07-20: 183 sections
2026-07-21: 232 sections
2026-07-23: 192 sections
2026-09-14: 236 sections
2026-09-15: 277 sections
2026-09-16: 277 sections
2026-11-09: 168 sections

Dates requested: 23
Dates retrieved: 23
Sections indexed: 5184
Failed dates: 0


In [ ]:
def normalize_title(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value).lower()).strip()


def cited_page_ranges(values):
    ranges = []

    for value in values:
        for match in re.finditer(
            r"\bH(\d+)(?:\s*[-–]\s*H?(\d+))?",
            str(value)
        ):
            start = int(match.group(1))
            end_text = match.group(2)

            if end_text is None:
                end = start
            else:
                # Handle shortened ranges such as H5599-602
                if len(end_text) < len(str(start)):
                    end_text = str(start)[:-len(end_text)] + end_text

                end = int(end_text)

                if end < start:
                    raise ValueError(
                        f"Check this page range: {match.group(0)}"
                    )

            ranges.append((start, end))

    return ranges


# Identify House sections and their starting pages
house_index = record_index.copy()

house_index["start_page"] = pd.to_numeric(
    house_index["document_id"].str.extract(
        r"-PgH(\d+)", expand=False
    ),
    errors="coerce"
)

house_index = house_index.dropna(subset=["start_page"]).copy()
house_index["start_page"] = house_index["start_page"].astype(int)

bill_titles = bills.set_index("Congress_Bill_ID")["Name"].to_dict()
matched_rows = []

for _, item in candidate_dates.iterrows():
    bill_id = item["Congress_Bill_ID"]
    date = item["date"]

    actions_for_date = candidate_actions[
        (candidate_actions["Congress_Bill_ID"] == bill_id)
        & (candidate_actions["date"] == date)
    ]

    page_ranges = cited_page_ranges(actions_for_date["House_pages"])
    expected_title = normalize_title(bill_titles[bill_id])

    sections_for_date = house_index[house_index["date"] == date]

    for _, section in sections_for_date.iterrows():
        page_match = any(
            start <= section["start_page"] <= end
            for start, end in page_ranges
        )

        title_match = (
            bool(expected_title)
            and normalize_title(section["title"]) == expected_title
        )

        if page_match or title_match:
            matched_rows.append({
                "Congress_Bill_ID": bill_id,
                "date": date,
                "document_id": section["document_id"],
                "title": section["title"],
                "page_match": page_match,
                "title_match": title_match
            })

candidate_documents = pd.DataFrame(
    matched_rows,
    columns=[
        "Congress_Bill_ID", "date", "document_id",
        "title", "page_match", "title_match"
    ]
).drop_duplicates()

# Keep unmatched bill-date combinations visible
unmatched_dates = candidate_dates.merge(
    candidate_documents[["Congress_Bill_ID", "date"]].drop_duplicates(),
    on=["Congress_Bill_ID", "date"],
    how="left",
    indicator=True
)

unmatched_dates = unmatched_dates[
    unmatched_dates["_merge"] == "left_only"
].drop(columns="_merge")

candidate_documents.to_csv(
    "/content/candidate_documents.csv", index=False
)

print("Bill-date combinations matched:",
      len(candidate_dates) - len(unmatched_dates))
print("Bill-document links:", len(candidate_documents))
print("Unique documents:", candidate_documents["document_id"].nunique())
print("Bill-date combinations needing review:", len(unmatched_dates))

display(unmatched_dates)

Bill-date combinations matched: 37
Bill-document links: 149
Unique documents: 147
Bill-date combinations needing review: 5


,Congress_Bill_ID,date
20,119-hr-8029,2026-03-24
23,119-hr-8646,2026-06-03
31,119-hr-8884,2026-06-30
32,119-hr-8884,2026-07-20
40,119-hr-9770,2026-07-20


In [ ]:
for _, item in unmatched_dates.iterrows():
    bill_id = item["Congress_Bill_ID"]
    date = item["date"]

    print("=" * 80)
    print(f"{bill_id} — {date}")
    print("Bill title:", bill_titles[bill_id])
    print()

    relevant_actions = candidate_actions[
        (candidate_actions["Congress_Bill_ID"] == bill_id)
        & (candidate_actions["date"] == date)
    ]

    for _, action in relevant_actions.iterrows():
        print("Action:", action["action_text"])
        print("House pages:", action["House_pages"] or "None recorded")
        print()

119-hr-8029 — 2026-03-24
Bill title: Pay Our Homeland Defenders Act

Action: Rules Committee Resolution H. Res. 1131 Reported to House. Rule provides for consideration of H.R. 8029, H. Res. 1128, H.R. 5103 and H.R. 7084. The resolution provides for consideration of H.R. 8029, H. Res. 1128, H.R. 5103, and H.R. 7084 under a closed rule with one hour of general debate on each measure. The resolution provides for one motion to recommit on H.R. 8029, H.R. 5103, and H.R. 7084.
House pages: None recorded

119-hr-8646 — 2026-06-03
Bill title: Agriculture, Rural Development, Food and Drug Administration, and Related Agency Appropriations Act, 2027

Action: Rules Committee Resolution H. Res. 1333 Reported to House. Rule provides for consideration of H.R. 8646, H.R. 7726, H.R. 7892 and H.R. 8872. The resolution provides for consideration of H.R. 8646 under a structured rule and H.R. 7726, H.R. 7892, and H.R. 8872 under a closed rule, with one hour of general debate on each bill. The resolution pr

In [ ]:
# Identify actions reporting a Rules Committee resolution
rule_report_mask = candidate_actions["action_text"].str.contains(
    r"^Rules Committee Resolution\b.*?\bReported to House\b",
    case=False,
    regex=True,
    na=False
)

rule_report_actions = candidate_actions[rule_report_mask].copy()
debate_candidate_actions = candidate_actions[~rule_report_mask].copy()

# Rebuild candidate dates from the remaining evidence
debate_candidate_dates = (
    debate_candidate_actions[["Congress_Bill_ID", "date"]]
    .drop_duplicates()
    .sort_values(["Congress_Bill_ID", "date"])
    .reset_index(drop=True)
)

# Retain document matches only for those dates
debate_candidate_documents = candidate_documents.merge(
    debate_candidate_dates,
    on=["Congress_Bill_ID", "date"],
    how="inner"
)

# Check whether any remaining dates still lack documents
remaining_unmatched = debate_candidate_dates.merge(
    debate_candidate_documents[
        ["Congress_Bill_ID", "date"]
    ].drop_duplicates(),
    on=["Congress_Bill_ID", "date"],
    how="left",
    indicator=True
)

remaining_unmatched = remaining_unmatched[
    remaining_unmatched["_merge"] == "left_only"
].drop(columns="_merge")

# Save both the retained candidates and the separate rule-report log
rule_report_actions.to_csv(
    "/content/rule_report_actions.csv", index=False
)
debate_candidate_dates.to_csv(
    "/content/debate_candidate_dates.csv", index=False
)
debate_candidate_documents.to_csv(
    "/content/debate_candidate_documents.csv", index=False
)

print("Bills with remaining candidate dates:",
      debate_candidate_dates["Congress_Bill_ID"].nunique())
print("Remaining bill-date combinations:", len(debate_candidate_dates))
print("Unique candidate documents:",
      debate_candidate_documents["document_id"].nunique())
print("Remaining unmatched dates:", len(remaining_unmatched))

display(remaining_unmatched)

Bills with remaining candidate dates: 36
Remaining bill-date combinations: 37
Unique candidate documents: 147
Remaining unmatched dates: 0


,Congress_Bill_ID,date


In [ ]:
from bs4 import BeautifulSoup

document_folder = Path("/content/record_documents")
document_folder.mkdir(exist_ok=True)

download_log = []

unique_documents = debate_candidate_documents[
    ["date", "document_id"]
].drop_duplicates("document_id")

for _, document in unique_documents.iterrows():
    date = document["date"]
    document_id = document["document_id"]

    source_url = (
        f"https://www.govinfo.gov/content/pkg/CREC-{date}/html/"
        f"{document_id}.htm"
    )

    html_file = document_folder / f"{document_id}.html"
    text_file = document_folder / f"{document_id}.txt"

    try:
        # Reuse downloaded originals when rerunning this cell
        if html_file.exists():
            html_content = html_file.read_bytes()
        else:
            response = requests.get(source_url, timeout=30)

            if response.status_code != 200:
                raise RuntimeError(
                    f"HTTP status {response.status_code}"
                )

            html_content = response.content
            time.sleep(0.5)

        soup = BeautifulSoup(html_content, "html.parser")
        record_body = soup.find("pre")

        if record_body is None:
            raise RuntimeError("Expected Record text not found")

        document_text = record_body.get_text()

        if not document_text.strip():
            raise RuntimeError("Record text is empty")

        html_file.write_bytes(html_content)
        text_file.write_text(document_text, encoding="utf-8")

        download_log.append({
            "document_id": document_id,
            "date": date,
            "source_url": source_url,
            "status": "saved",
            "characters": len(document_text)
        })

        print(f"Saved: {document_id}")

    except Exception as error:
        reason = (
            str(error)
            if isinstance(error, RuntimeError)
            else type(error).__name__
        )

        download_log.append({
            "document_id": document_id,
            "date": date,
            "source_url": source_url,
            "status": "failed",
            "reason": reason
        })

        print(f"Failed: {document_id} — {reason}")

download_log = pd.DataFrame(download_log)
download_log.to_csv("/content/document_download_log.csv", index=False)

print("\nDownload summary:")
print(download_log["status"].value_counts())

display(download_log[download_log["status"] == "failed"])

Saved: CREC-2026-09-14-pt1-PgH5599
Saved: CREC-2026-09-14-pt1-PgH5599-2
Saved: CREC-2026-09-14-pt1-PgH5602
Saved: CREC-2026-07-15-pt1-PgH4472
Saved: CREC-2026-07-15-pt1-PgH4472-2
Saved: CREC-2026-07-15-pt1-PgH4473
Saved: CREC-2026-04-14-pt1-PgH2852
Saved: CREC-2026-04-14-pt1-PgH2853
Saved: CREC-2026-04-14-pt1-PgH2853-2
Saved: CREC-2026-06-02-pt1-PgH3754
Saved: CREC-2026-06-02-pt1-PgH3755
Saved: CREC-2026-07-20-pt1-PgH4651
Saved: CREC-2026-07-20-pt1-PgH4652
Saved: CREC-2026-04-14-pt1-PgH2846
Saved: CREC-2026-04-14-pt1-PgH2847
Saved: CREC-2026-07-13-pt1-PgH4386
Saved: CREC-2026-07-13-pt1-PgH4386-2
Saved: CREC-2026-07-13-pt1-PgH4386-3
Saved: CREC-2026-07-13-pt1-PgH4386-4
Saved: CREC-2026-07-13-pt1-PgH4387
Saved: CREC-2026-06-08-pt1-PgH3968
Saved: CREC-2026-06-08-pt1-PgH3970
Saved: CREC-2026-03-24-pt1-PgH2648
Saved: CREC-2026-03-24-pt1-PgH2649
Saved: CREC-2026-03-24-pt1-PgH2650
Saved: CREC-2026-07-14-pt1-PgH4438
Saved: CREC-2026-07-14-pt1-PgH4441
Saved: CREC-2026-07-14-pt1-PgH4441-2
Saved:

,document_id,date,source_url,status,characters


In [ ]:
from bs4 import BeautifulSoup

document_folder = Path("/content/record_documents")
document_folder.mkdir(exist_ok=True)

download_log = []

unique_documents = debate_candidate_documents[
    ["date", "document_id"]
].drop_duplicates("document_id")

for _, document in unique_documents.iterrows():
    date = document["date"]
    document_id = document["document_id"]

    source_url = (
        f"https://www.govinfo.gov/content/pkg/CREC-{date}/html/"
        f"{document_id}.htm"
    )

    html_file = document_folder / f"{document_id}.html"
    text_file = document_folder / f"{document_id}.txt"

    try:
        # Reuse downloaded originals when rerunning this cell
        if html_file.exists():
            html_content = html_file.read_bytes()
        else:
            response = requests.get(source_url, timeout=30)

            if response.status_code != 200:
                raise RuntimeError(
                    f"HTTP status {response.status_code}"
                )

            html_content = response.content
            time.sleep(0.5)

        soup = BeautifulSoup(html_content, "html.parser")
        record_body = soup.find("pre")

        if record_body is None:
            raise RuntimeError("Expected Record text not found")

        document_text = record_body.get_text()

        if not document_text.strip():
            raise RuntimeError("Record text is empty")

        html_file.write_bytes(html_content)
        text_file.write_text(document_text, encoding="utf-8")

        download_log.append({
            "document_id": document_id,
            "date": date,
            "source_url": source_url,
            "status": "saved",
            "characters": len(document_text)
        })

        print(f"Saved: {document_id}")

    except Exception as error:
        reason = (
            str(error)
            if isinstance(error, RuntimeError)
            else type(error).__name__
        )

        download_log.append({
            "document_id": document_id,
            "date": date,
            "source_url": source_url,
            "status": "failed",
            "reason": reason
        })

        print(f"Failed: {document_id} — {reason}")

download_log = pd.DataFrame(download_log)
download_log.to_csv("/content/document_download_log.csv", index=False)

print("\nDownload summary:")
print(download_log["status"].value_counts())

display(download_log[download_log["status"] == "failed"])

Saved: CREC-2026-09-14-pt1-PgH5599
Saved: CREC-2026-09-14-pt1-PgH5599-2
Saved: CREC-2026-09-14-pt1-PgH5602
Saved: CREC-2026-07-15-pt1-PgH4472
Saved: CREC-2026-07-15-pt1-PgH4472-2
Saved: CREC-2026-07-15-pt1-PgH4473
Saved: CREC-2026-04-14-pt1-PgH2852
Saved: CREC-2026-04-14-pt1-PgH2853
Saved: CREC-2026-04-14-pt1-PgH2853-2
Saved: CREC-2026-06-02-pt1-PgH3754
Saved: CREC-2026-06-02-pt1-PgH3755
Saved: CREC-2026-07-20-pt1-PgH4651
Saved: CREC-2026-07-20-pt1-PgH4652
Saved: CREC-2026-04-14-pt1-PgH2846
Saved: CREC-2026-04-14-pt1-PgH2847
Saved: CREC-2026-07-13-pt1-PgH4386
Saved: CREC-2026-07-13-pt1-PgH4386-2
Saved: CREC-2026-07-13-pt1-PgH4386-3
Saved: CREC-2026-07-13-pt1-PgH4386-4
Saved: CREC-2026-07-13-pt1-PgH4387
Saved: CREC-2026-06-08-pt1-PgH3968
Saved: CREC-2026-06-08-pt1-PgH3970
Saved: CREC-2026-03-24-pt1-PgH2648
Saved: CREC-2026-03-24-pt1-PgH2649
Saved: CREC-2026-03-24-pt1-PgH2650
Saved: CREC-2026-07-14-pt1-PgH4438
Saved: CREC-2026-07-14-pt1-PgH4441
Saved: CREC-2026-07-14-pt1-PgH4441-2
Saved:

,document_id,date,source_url,status,characters


In [ ]:
bill_document_checks = []

for _, document in debate_candidate_documents.iterrows():
    bill_id = document["Congress_Bill_ID"]
    document_id = document["document_id"]
    bill_number = bill_id.split("-")[-1]

    text = (
        document_folder / f"{document_id}.txt"
    ).read_text(encoding="utf-8")

    # Match H.R. 7436, H. R. 7436, HR 7436, etc.
    pattern = re.compile(
        rf"\bH\.?\s*R\.?\s*{re.escape(bill_number)}\b",
        flags=re.IGNORECASE
    )

    mentions = list(pattern.finditer(text))

    # Show context around the first mention
    if mentions:
        first = mentions[0]
        excerpt = text[
            max(0, first.start() - 150):first.end() + 350
        ]
        excerpt = " ".join(excerpt.split())
    else:
        excerpt = ""

    bill_document_checks.append({
        "Congress_Bill_ID": bill_id,
        "date": document["date"],
        "document_id": document_id,
        "title": document["title"],
        "bill_mentions": len(mentions),
        "title_match": document["title_match"],
        "first_mention_context": excerpt
    })

bill_document_checks = pd.DataFrame(bill_document_checks)

bill_document_checks.to_csv(
    "/content/bill_document_checks.csv", index=False
)

# Summarise coverage for each expected bill-date combination
mention_counts = (
    bill_document_checks.assign(
        has_mention=bill_document_checks["bill_mentions"].gt(0)
    )
    .groupby(["Congress_Bill_ID", "date"], as_index=False)
    .agg(documents_with_bill_number=("has_mention", "sum"))
)

mention_coverage = debate_candidate_dates.merge(
    mention_counts,
    on=["Congress_Bill_ID", "date"],
    how="left"
)

mention_coverage["documents_with_bill_number"] = (
    mention_coverage["documents_with_bill_number"]
    .fillna(0)
    .astype(int)
)

needs_review = mention_coverage[
    mention_coverage["documents_with_bill_number"] == 0
]

print("Bill-document links checked:", len(bill_document_checks))
print("Links containing the bill number:",
      bill_document_checks["bill_mentions"].gt(0).sum())
print("Bill-date combinations without a bill-number match:",
      len(needs_review))

display(needs_review)

Bill-document links checked: 149
Links containing the bill number: 46
Bill-date combinations without a bill-number match: 1


,Congress_Bill_ID,date,documents_with_bill_number
24,119-hr-8770,2026-07-13,0


In [ ]:
review_bill = "119-hr-8770"
review_date = "2026-07-13"

print("LEGISLATIVE ACTIONS\n")

relevant_actions = debate_candidate_actions[
    (debate_candidate_actions["Congress_Bill_ID"] == review_bill)
    & (debate_candidate_actions["date"] == review_date)
]

for _, action in relevant_actions.iterrows():
    print(action["action_text"])
    print("House pages:", action["House_pages"])
    print()

print("\nCANDIDATE DOCUMENTS\n")

review_documents = bill_document_checks[
    (bill_document_checks["Congress_Bill_ID"] == review_bill)
    & (bill_document_checks["date"] == review_date)
]

for _, document in review_documents.iterrows():
    text = (
        document_folder / f"{document['document_id']}.txt"
    ).read_text(encoding="utf-8")

    print("=" * 80)
    print("Title:", document["title"])
    print("Document:", document["document_id"])
    print("Exact title match:", document["title_match"])
    print("\nBeginning of document:\n")
    print(text[:2000])
    print()

LEGISLATIVE ACTIONS

On motion to suspend the rules and pass the bill, as amended Agreed to by voice vote. (text: CR H4389-4390)
House pages: H4389-4390

Passed/agreed to in House: On motion to suspend the rules and pass the bill, as amended Agreed to by voice vote. (text: CR H4389-4390)
House pages: H4389-4390

DEBATE - The House proceeded with forty minutes of debate on H.R. 8770.
House pages: 

Considered under suspension of the rules. (consideration: CR H4389-4390)
House pages: H4389-4390


CANDIDATE DOCUMENTS

Title: IMPROVING TRAVEL FOR MILITARY MEMBERS ACT
Document: CREC-2026-07-13-pt1-PgH4390
Exact title match: False

Beginning of document:


[Congressional Record Volume 172, Number 113 (Monday, July 13, 2026)]
[House]
[Pages H4390-H4391]
From the Congressional Record Online through the Government Publishing Office [www.gpo.gov]




               IMPROVING TRAVEL FOR MILITARY MEMBERS ACT

  Mr. GARBARINO. Mr. Speaker, I move to suspend the rules and pass the 
bill (H.R. 9328) 

In [ ]:
title_candidates = record_index[
    record_index["date"].eq("2026-07-13")
    & record_index["title"].str.contains(
        "safeguard", case=False, na=False
    )
]

print("Matching section titles:", len(title_candidates))

for _, section in title_candidates.iterrows():
    print("Title:", section["title"])
    print("Document ID:", section["document_id"])
    print()

Matching section titles: 0


In [ ]:
excluded_bill = "119-hr-8770"

exclusion_log = pd.DataFrame([{
    "Congress_Bill_ID": excluded_bill,
    "reason": "Unresolved match between bill history and debate document"
}])

active_bills = bills[
    bills["Congress_Bill_ID"] != excluded_bill
].copy()

active_document_links = bill_document_checks[
    bill_document_checks["Congress_Bill_ID"] != excluded_bill
].copy()

exclusion_log.to_csv("/content/excluded_bills.csv", index=False)

print("Bills remaining:", len(active_bills))
print(active_bills["Sponsor Party"].value_counts())

Bills remaining: 35
Sponsor Party
Democratic    18
Republican    17
Name: count, dtype: int64


In [ ]:
matched_document_links = active_document_links[
    active_document_links["bill_mentions"] > 0
].copy()

matched_document_links = matched_document_links.sort_values(
    ["Congress_Bill_ID", "date", "document_id"]
).reset_index(drop=True)

for index, document in matched_document_links.iterrows():
    print("=" * 80)
    print(f"Candidate {index + 1}")
    print("Bill:", document["Congress_Bill_ID"])
    print("Date:", document["date"])
    print("Title:", document["title"])
    print("Document:", document["document_id"])
    print("Bill mentions:", document["bill_mentions"])
    print("Context:", document["first_mention_context"])
    print()

matched_document_links.to_csv(
    "/content/matched_document_links.csv", index=False
)

print("Candidate links:", len(matched_document_links))
print("Bills represented:",
      matched_document_links["Congress_Bill_ID"].nunique())

Candidate 1
Bill: 119-hr-10167
Date: 2026-09-14
Title: COMMON CENTS ACT
Document: CREC-2026-09-14-pt1-PgH5599-2
Bill mentions: 6
Context: {time} 1910 COMMON CENTS ACT Ms. DE LA CRUZ. Mr. Speaker, I move to suspend the rules and pass the bill (H.R. 10167) to direct the Secretary of the Treasury to stop minting the penny, to require cash transactions to be rounded up or down to the nearest 5 cents, and for other purposes. The Clerk read the title of the bill. The text of the bill is as follows: H.R. 10167 Be it enacted by the Senate and House of Represe

Candidate 2
Bill: 119-hr-6007
Date: 2026-07-15
Title: ANTHONY "TONY" J. GALLOPO VA CLINIC
Document: CREC-2026-07-15-pt1-PgH4472-2
Bill mentions: 9
Context: ce [www.gpo.gov] ANTHONY ``TONY'' J. GALLOPO VA CLINIC Mr. BOST. Mr. Speaker, I move to suspend the rules and pass the bill (H.R. 6007) to name the Department of Veterans Affairs community-based outpatient clinic in Newton, New Jersey, as the ``Anthony `Tony' J. Gallopo VA Clinic''. T

In [1]:
# Recognise members and presiding officers
speaker_pattern = re.compile(
    r"^[ \t]*(?P<speaker>"
    r"(?:Mr\.|Ms\.|Mrs\.)[ \t]+[A-Z][A-Za-z '\u2019—-]*"
    r"|The (?:SPEAKER|CHAIRMAN|CHAIRWOMAN|CHAIR)"
    r"(?: pro tempore)?(?:[ \t]+\([^\n)]*\))?"
    r")\.[ \t]+",
    re.MULTILINE
)

all_turns = []
extraction_log = []

# Extract each document once, even if linked to multiple bills
documents_to_split = matched_document_links.drop_duplicates(
    "document_id"
)

for _, document in documents_to_split.iterrows():
    document_id = document["document_id"]

    text = (
        document_folder / f"{document_id}.txt"
    ).read_text(encoding="utf-8")

    labels = list(speaker_pattern.finditer(text))

    extraction_log.append({
        "document_id": document_id,
        "title": document["title"],
        "candidate_turns": len(labels),
        "characters_before_first_label": (
            labels[0].start() if labels else len(text)
        )
    })

    for index, label in enumerate(labels):
        end = (
            labels[index + 1].start()
            if index + 1 < len(labels)
            else len(text)
        )

        turn_text = text[label.end():end].strip()

        all_turns.append({
            "segment_id": f"{document_id}_t{index + 1:04d}",
            "document_id": document_id,
            "date": document["date"],
            "turn_number": index + 1,
            "speaker_raw": label.group("speaker"),
            "text_raw": turn_text,
            "character_count": len(turn_text),
            "label_start": label.start(),
            "text_start": label.end(),
            "segment_end": end,
            "review_status": "not_reviewed"
        })

all_candidate_turns = pd.DataFrame(all_turns)
extraction_log = pd.DataFrame(extraction_log)

all_candidate_turns.to_csv(
    "/content/all_candidate_turns.csv", index=False
)
extraction_log.to_csv(
    "/content/speaker_extraction_log.csv", index=False
)

# Preserve the separate links between bills and documents
matched_document_links.to_csv(
    "/content/bill_document_links.csv", index=False
)

print("Documents processed:", len(extraction_log))
print("Candidate turns extracted:", len(all_candidate_turns))
print("Documents with no detected speakers:",
      extraction_log["candidate_turns"].eq(0).sum())

display(
    extraction_log[extraction_log["candidate_turns"] == 0]
)

print("\nTen longest segments — useful for detecting embedded bill text:")
display(
    all_candidate_turns.nlargest(10, "character_count")[
        ["segment_id", "speaker_raw", "character_count"]
    ]
)

NameError: name 're' is not defined

In [2]:
import re
import json
import time
import pandas as pd
from pathlib import Path

print("Bill-document table available:",
      "matched_document_links" in globals())

print("Downloaded text files:",
      len(list(Path("/content/record_documents").glob("*.txt"))))

Bill-document table available: False
Downloaded text files: 0


In [3]:
from google.colab import files

uploaded = files.upload()

Saving Bills 1.xlsx to Bills 1.xlsx


In [4]:
import pandas as pd
import re
import json
import time
import requests
from pathlib import Path

bills = pd.read_excel("/content/Bills 1.xlsx")
bills = bills.rename(columns={"Unnamed: 1": "Bill_Number"})

bills = bills[
    bills["Bill_Number"].notna()
    & bills["Bill_Number"].astype(str).str.strip().ne("")
].copy()

bill_numbers = bills["Bill_Number"].astype(str).str.extract(
    r"^\s*H\s*\.\s*R\s*\.\s*(\d+)\s*$",
    expand=False
)

if bill_numbers.isna().any():
    display(bills.loc[bill_numbers.isna(), ["Bill_ID", "Bill_Number"]])
    raise ValueError("Check the bill numbers displayed above.")

bills["Congress_Bill_ID"] = "119-hr-" + bill_numbers

if bills["Congress_Bill_ID"].duplicated().any():
    raise ValueError("Duplicate bill identifiers found.")

active_bills = bills[
    bills["Congress_Bill_ID"] != "119-hr-8770"
].copy()

print("Bills in workbook:", len(bills))
print("Active bills:", len(active_bills))
print(active_bills["Sponsor Party"].value_counts())

Bills in workbook: 36
Active bills: 35
Sponsor Party
Democratic    18
Republican    17
Name: count, dtype: int64


In [5]:
from google.colab import drive

drive.mount("/content/drive")

project_folder = Path(
    "/content/drive/MyDrive/PhD_US_Bill_Speeches"
)
project_folder.mkdir(parents=True, exist_ok=True)

# Save the bill lists
bills.to_csv(project_folder / "all_bills.csv", index=False)
active_bills.to_csv(project_folder / "active_bills.csv", index=False)

# Create permanent folders for the downloaded material
history_folder = project_folder / "bill_histories"
document_folder = project_folder / "record_documents"
section_folder = project_folder / "record_section_lists"

for folder in [history_folder, document_folder, section_folder]:
    folder.mkdir(exist_ok=True)

print("Collection folder:", project_folder)
print("Bill lists saved to Google Drive.")

Mounted at /content/drive
Collection folder: /content/drive/MyDrive/PhD_US_Bill_Speeches
Bill lists saved to Google Drive.


In [7]:
congress_api_key = getpass(RfNGmBMCjTwfDjFptcjhgCqhcPq8z6tRBnPRTHzQ).strip()

NameError: name 'getpass' is not defined

In [9]:
from getpass import getpass

congress_api_key = getpass("Paste your API key here: ").strip()

print("Key stored.")

Paste your API key here: ··········
Key stored.
